# Functional Decoding Word Clouds

This notebook aims to visualize the functional decoding results of our subregions using custom word clouds. Instead of presenting simple bar charts or tables, we map the highest-ranking meta-analytic cognitive terms (derived from Neurosynth) onto circular word clouds.

The size of each term reflects its relevance score (normalized $p(F|A)$), and the color of the term is dynamically assigned based on predefined cognitive domains (e.g., Valuation, Social, Emotion) to provide a clear, intuitive summary of each subregion's primary functional profile.


To ensure the word clouds only display meaningful cognitive concepts, we first load a curated term list (`term_selection.xls`) to filter out unspecific or purely anatomical terms.

We then define custom functions to categorize and color-code the remaining terms. Consistent with our global aesthetic, we map:
* **Blue (`#2b60ff`)** for Social cognition terms.
* **Red (`#cc3300`)** for Emotion / Affective terms.
* **Yellow (`#dda700`)** for Valuation / Reward terms.
* **Black (`#000000`)** for other non-specific cognitive terms.

In [ ]:
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import minmax_scale
from pathlib import Path
from wordcloud import WordCloud


plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'Arial'

DATA_PATH = Path('../data')
RESULTS_PATH = Path('../results')
PLOT_PATH = Path('../plots')
PLOT_PARAMS = dict(bbox_inches='tight', transparent=True, dpi=600)
N_CLUSTER_LIST = list(range(2, 7))
RED, YELLOW, BLUE = '#c00000', '#dcd844', '#0033cc'
RED, YELLOW, BLUE, ORANGE, GREEN, GOLD = ('#c00000', '#dcd844', '#0033cc', '#FF5733', '#4CAF50', '#FFD700')
DARK_RED, DARK_YELLOW, DARK_BLUE, = ['#8b0000', '#e6af00', '#002699', ]
CANVAS_WIDTH, CANVAS_HEIGHT = 2000, 2000

In [ ]:
circle_mask = np.ones((CANVAS_HEIGHT, CANVAS_WIDTH), dtype=np.uint8) * 255
center_x, center_y = CANVAS_WIDTH // 2, CANVAS_HEIGHT // 2
radius = min(CANVAS_WIDTH, CANVAS_HEIGHT) // 3
Y, X = np.ogrid[:CANVAS_HEIGHT, :CANVAS_WIDTH]
mask = (X - center_x) ** 2 + (Y - center_y) ** 2 <= radius ** 2
circle_mask[mask] = 0

In [ ]:
term_select_df = pd.read_excel(DATA_PATH / 'neurosynth_data/term_selection.xls')
excluded_term_list = term_select_df.query('(type == "unspecific") or (type == "anatomy") ')['term'].tolist()
social_term_list = term_select_df.query('type.str.contains("social")')['term'].tolist()
emotion_term_list = term_select_df.query('type.str.contains("affect")')['term'].tolist()
valuation_term_list = term_select_df.query('type.str.contains("valuation")')['term'].tolist()
others_term_list = term_select_df.query('type.str.contains("others")')['term'].tolist()



def color_func(word, font_size, position, orientation, random_state=None, **kwargs):
    word = word.lower()
    if word in social_term_list:
        return "#2b60ff"  # blue
    elif word in emotion_term_list:
        return "#cc3300"  # red
    elif word in valuation_term_list:
        return "#dda700"  # yellow
    elif word in others_term_list:
        return "#000000"  # black

    return GREEN  # fallback color for unmapped terms


def term_map(term):
    return term.title().replace('Tom', 'ToM').replace('Asd', 'ASD')


def group_of(word: str):
    w = word.lower()
    if w in social_term_list:
        return 'social'
    if w in emotion_term_list:
        return 'emotion'
    if w in valuation_term_list:
        return 'valuation'
    return 'other'

# Plot

Finally, we iterate through the decoding results for each cluster. For each file, we extract the top 15 most relevant cognitive terms. The term scores are normalized to dictate font sizes. We generate high-resolution, circular word clouds with transparent backgrounds, applying our custom color mapping to visually emphasize the dominant functional network of the specific cluster.

In [ ]:
data_folder = RESULTS_PATH / 'NSplus_rank'
words = []
files = sorted(list(data_folder.iterdir()))
for file in files:

    df = pd.read_csv(file, index_col=0, skiprows=2)
    plot_data = df.query('term not in @excluded_term_list').iloc[:15].set_index('term')
    if ('autobiographical memory' in plot_data.index) and ('autobiographical' in plot_data.index):
        plot_data = df.query('term not in @excluded_term_list').iloc[:16].set_index('term')
        plot_data.drop('autobiographical memory', axis=0, inplace=True)

    plot_data = plot_data[['pFgA_given_pF=0.50']]
    plot_data.index = plot_data.index.map(term_map)
    plot_data['score'] = minmax_scale(plot_data['pFgA_given_pF=0.50'])
    words += plot_data.index.tolist()
    word_freq = (plot_data['score'] * 0.5 + 5).to_dict()

    word_color = plot_data['score'].to_dict()
    wordcloud = WordCloud(
        width=CANVAS_WIDTH,
        height=CANVAS_HEIGHT,
        mask=circle_mask,
        background_color="rgba(255, 255, 255, 0)",
        mode="RGBA",
        color_func=color_func,
        prefer_horizontal=1,
        margin=50,
        max_font_size=200,
    ).generate_from_frequencies(word_freq)

    fig, ax = plt.subplots()
    ax.imshow(wordcloud, interpolation="bilinear")
    ax.axis("off")
    fig.savefig(PLOT_PATH / f'wordcloud/{file.stem}.svg', **PLOT_PARAMS)
    fig.savefig(PLOT_PATH / f'wordcloud/{file.stem}.png', **PLOT_PARAMS)